# IMAGE SEGMENTATION

In [ ]:
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz
!tar -xf images.tar.gz
!tar -xf annotations.tar.gz

In [ ]:
!pip install -q git+https://github.com/tensorflow/examples.git

In [ ]:
import os
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import clear_output
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import load_img, img_to_array, array_to_img, plot_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import tensorflow_datasets as tfds
from tensorflow_examples.models.pix2pix import pix2pix
from tensorflow.keras.applications import MobileNetV2
import gc
import cv2 as cv

In [ ]:
input_dir = "/kaggle/working/images"
target_dir = "/kaggle/working/annotations/trimaps/"

input_img_paths = sorted(
    [os.path.join(input_dir, fname)
     for fname in os.listdir(input_dir)
     if fname.endswith(".jpg")]
)

target_paths = sorted(
    [os.path.join(target_dir, fname)
    for fname in os.listdir(target_dir)
    if fname.endswith(".png")
    and not fname.startswith(".")]
)

In [ ]:
plt.axis('off')
plt.imshow(load_img(input_img_paths[9]))

In [ ]:
def display_target(target_array):
    normalized_array = (target_array.astype("uint8") - 1) * 127
    plt.axis('off')
    if len(normalized_array.shape) == 2:  # 2D array
        plt.imshow(normalized_array)
    elif len(normalized_array.shape) == 3:  # 3D array
        plt.imshow(normalized_array[:, :, 0])

img = img_to_array(load_img(target_paths[9], color_mode="grayscale"))
display_target(img)

In [ ]:
for i in range(10):
    img = img_to_array(load_img(input_img_paths[i]))
    print(img.shape)

In [ ]:
img_size = (200, 200)

num_imgs = len(input_img_paths)

rng = random.Random(1337)
paired_paths = list(zip(input_img_paths, target_paths))
rng.shuffle(paired_paths)
input_img_paths, target_paths = zip(*paired_paths)
input_img_paths = list(input_img_paths)
target_paths = list(target_paths)

def path_to_input_img(path):
    return cv.resize(img_to_array(load_img(path)), img_size)

def path_to_target_img(path):
    img = cv.resize(img_to_array(load_img(path, color_mode="grayscale")), img_size)
    # img = np.expand_dims(img, axis=-1)
    return img.astype("uint8") - 1

input_imgs = np.zeros((num_imgs,) + img_size + (3,))
targets = np.zeros((num_imgs,) + img_size)
for i in range(num_imgs):
    input_imgs[i] = path_to_input_img(input_img_paths[i])
    targets[i] = path_to_target_img(target_paths[i])

num_val_samples = 1000
train_input_images = input_imgs[:-num_val_samples]
train_targets = targets[:-num_val_samples]
val_input_images = input_imgs[-num_val_samples:]
val_targets = targets[-num_val_samples:]

In [ ]:
def get_model(img_size, num_classes):
    inputs = keras.Input(shape=img_size + (3,))  # Input layer (RGB image)
    x = layers.Rescaling(1./255)(inputs)  # Rescaling the image

    # Encoder Block (Feature extraction with 6 Conv2D layers)
    x = layers.Conv2D(32, kernel_size=3, strides=1, activation='relu', padding='same')(x)
    x = layers.Conv2D(32, kernel_size=3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2D(32, kernel_size=3, strides=1, activation='relu', padding='same')(x)
    x = layers.Conv2D(32, kernel_size=3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2D(32, kernel_size=3, strides=1, activation='relu', padding='same')(x)
    x = layers.Conv2D(32, kernel_size=3, strides=2, activation='relu', padding='same')(x)

    # Decoder Block (Upsampling with 6 Conv2DTranspose layers)
    x = layers.Conv2DTranspose(32, kernel_size=3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(32, kernel_size=3, strides=1, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(32, kernel_size=3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(32, kernel_size=3, strides=1, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(32, kernel_size=3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(32, kernel_size=3, strides=1, activation='relu', padding='same')(x)

    # Final segmentation output with softmax activation
    outputs = layers.Conv2D(num_classes, kernel_size=3, activation='softmax', padding='same')(x)

    # Create the Model
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Define image size (e.g., 200x200) and number of classes (e.g., 3 for segmentation)
img_size = (200, 200)
num_classes = 3

model = get_model(img_size=img_size, num_classes=num_classes)
model.summary()

In [ ]:
def get_model(img_size, num_classes):
    inputs = keras.Input(shape=img_size + (3,))  # Input layer (RGB image)
    x = layers.Rescaling(1./255)(inputs)  # Rescaling the image

    # Encoder Block (Feature extraction with 6 Conv2D layers)
    x = layers.Conv2D(32, kernel_size=3, strides=1, activation='relu', padding='same')(x)
    x = layers.Conv2D(64, kernel_size=3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2D(64, kernel_size=3, strides=1, activation='relu', padding='same')(x)
    x = layers.Conv2D(128, kernel_size=3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2D(128, kernel_size=3, strides=1, activation='relu', padding='same')(x)
    x = layers.Conv2D(256, kernel_size=3, strides=2, activation='relu', padding='same')(x)

    # Decoder Block (Upsampling with 6 Conv2DTranspose layers)
    x = layers.Conv2DTranspose(128, kernel_size=3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(128, kernel_size=3, strides=1, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(64, kernel_size=3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(64, kernel_size=3, strides=1, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(32, kernel_size=3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(32, kernel_size=3, strides=1, activation='relu', padding='same')(x)

    # Final segmentation output with softmax activation
    outputs = layers.Conv2D(num_classes, kernel_size=3, activation='softmax', padding='same')(x)

    # Create the Model
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

# Define image size (e.g., 200x200) and number of classes (e.g., 3 for segmentation)
img_size = (200, 200)
num_classes = 3

model3 = get_model(img_size=img_size, num_classes=num_classes)
model3.summary()

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.01), loss="sparse_categorical_crossentropy", metrics=['accuracy'])
callbacks1 = [
    keras.callbacks.ModelCheckpoint("oxford_segmentation.keras",
                                    save_best_only=True, verbose=2)
]

In [ ]:
model3.compile(optimizer=Adam(learning_rate=0.01), loss="sparse_categorical_crossentropy", metrics=['accuracy'])
callbacks = [
    keras.callbacks.ModelCheckpoint("oxford_segmentation3.keras",
                                    save_best_only=True, verbose=2)
]

In [ ]:
history1 = model.fit(x=train_input_images, y=train_targets,
                    epochs=3, batch_size=64,
                    validation_data=(val_input_images, val_targets), callbacks=callbacks1)

In [ ]:
history=model3.fit(x=train_input_images, y=train_targets,
                    epochs=3, batch_size=64,
                    validation_data=(val_input_images, val_targets), callbacks=callbacks)

In [ ]:
# Plotting Loss for model
epochs = range(1, len(history1.history["loss"]) + 1)
loss = history1.history["loss"]# history me se Loss nikalo
val_loss = history1.history["val_loss"]# history me se validation loss nikalo
plt.figure()
plt.plot(epochs, loss, "bo", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()

In [ ]:
# Plotting Loss for model3
epochs = range(1, len(history.history["loss"]) + 1)
loss = history.history["loss"]# history me se Loss nikalo
val_loss = history.history["val_loss"]# history me se validation loss nikalo
plt.figure()
plt.plot(epochs, loss, "bo", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()

In [ ]:
model = load_model("/kaggle/working/oxford_segmentation.keras")

In [ ]:
model3 = load_model("/kaggle/working/oxford_segmentation3.keras")

In [ ]:
# prediction from model3
def predict_and_display(i):
    test_image = val_input_images[i]
    target = val_targets[i]
    resized_image = np.expand_dims(test_image, axis=0)
    mask = model3.predict(resized_image)[0]
    return test_image, mask, target

def display_mask(pred):
    mask = np.argmax(pred, axis=-1)
    mask *= 127
    print(np.unique(mask))
    return mask

num_test_images = 20
fig, axes = plt.subplots(num_test_images, 3, figsize=(15, 5 * num_test_images))

for i in range(num_test_images):
    test_image, pred_mask, target = predict_and_display(i)
    
    axes[i, 0].imshow(array_to_img(test_image))
    axes[i, 0].axis("off")
    axes[i, 0].set_title(f"Test Image {i+1}")
    
    axes[i, 1].imshow(display_mask(pred_mask))
    axes[i, 1].axis("off")
    axes[i, 1].set_title(f"Predicted Mask {i+1}")

    target = np.expand_dims(target, axis=-1)
    axes[i, 2].imshow(array_to_img(target))
    axes[i, 2].axis("off")
    axes[i, 2].set_title(f"Expected Mask {i+1}")

plt.tight_layout()
plt.show()


In [ ]:
# prediction from model
def predict_and_display(i):
    test_image = val_input_images[i]
    target = val_targets[i]
    resized_image = np.expand_dims(test_image, axis=0)
    mask = model.predict(resized_image)[0]
    return test_image, mask, target

def display_mask(pred):
    mask = np.argmax(pred, axis=-1)
    print(np.unique(mask))
    mask *= 127
    return mask

num_test_images = 20
fig, axes = plt.subplots(num_test_images, 3, figsize=(10, 5 * num_test_images))

for i in range(num_test_images):
    test_image, pred_mask, target = predict_and_display(i)
    
    axes[i, 0].imshow(array_to_img(test_image))
    axes[i, 0].axis("off")
    axes[i, 0].set_title(f"Test Image {i+1}")
    
    axes[i, 1].imshow(display_mask(pred_mask))
    axes[i, 1].axis("off")
    axes[i, 1].set_title(f"Predicted Mask {i+1}")

    target = np.expand_dims(target, axis=-1)
    axes[i, 2].imshow(array_to_img(target))
    axes[i, 2].axis("off")
    axes[i, 2].set_title(f"Expected Mask {i+1}")

plt.tight_layout()
plt.show()


In [ ]:
del(train_input_images, train_targets, val_input_images, val_targets, history1, input_imgs, targets)
gc.collect()

In [ ]:
del(model3, history)
gc.collect()

# U-NET Model

In [ ]:
dataset, info = tfds.load('oxford_iiit_pet:3.*.*', with_info=True)

In [ ]:
def normalize(input_image, input_mask):
  input_image = tf.cast(input_image, tf.float32) / 255.0
  input_mask -= 1
  return input_image, input_mask

In [ ]:
def load_image(datapoint):
  input_image = tf.image.resize(datapoint["image"], (128, 128)) # Use tf.image to resize datapoint["image"] to (128,128)
  input_mask = tf.image.resize(datapoint["segmentation_mask"], (128, 128), method=tf.image.ResizeMethod.NEAREST_NEIGHBOR) # Use Nearest Neighbour Method to resize datapoint["segmentation_mask"] to (128,128) for the image mask

  input_image, input_mask = normalize(input_image, input_mask)# Normalize the Images

  return input_image, input_mask

In [ ]:
TRAIN_LENGTH = info.splits['train'].num_examples
BATCH_SIZE = 64
BUFFER_SIZE = 1000
STEPS_PER_EPOCH = TRAIN_LENGTH // BATCH_SIZE

In [ ]:
train_images = dataset['train'].map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_images = dataset['test'].map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
class Augment(tf.keras.layers.Layer):
    def __init__(self, seed=42):
        super().__init__()
        self.augment_inputs = tf.keras.layers.RandomFlip(mode="horizontal", seed=seed)
        self.augment_labels = tf.keras.layers.RandomFlip(mode="horizontal", seed=seed)

    def call(self, inputs, labels):
        inputs = self.augment_inputs(inputs)
        labels = self.augment_labels(labels)
        return inputs, labels

In [ ]:
random.Random(1337)

In [ ]:
train_batches = train_images.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE).repeat().map(Augment()).prefetch(buffer_size = tf.data.AUTOTUNE)

test_batches = test_images.batch(BATCH_SIZE)

In [ ]:
def display(display_list):
  plt.figure(figsize=(15, 15))

  title = ['Input Image', 'True Mask', 'Predicted Mask']

  for i in range(len(display_list)):
    plt.subplot(1, len(display_list), i+1)
    plt.title(title[i])
    plt.imshow(tf.keras.utils.array_to_img(display_list[i]))
    plt.axis('off')
  plt.show()

In [ ]:
for image, mask in train_batches.take(2):
    input_image = image[0]
    input_mask = mask[0]
    display([input_image, input_mask])

In [ ]:
# Load the MobileNetV2 model without its final Dense layers (include_top=False)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# Define the layers from which to extract activations
layer_names = [
    'block_1_expand_relu',   # 64x64
    'block_3_expand_relu',   # 32x32
    'block_6_expand_relu',   # 16x16
    'block_13_expand_relu',  # 8x8
    'block_16_project',      # 4x4
]

# Get the outputs of the specified layers
base_model_outputs = [base_model.get_layer(name).output for name in layer_names]

# Create the feature extraction model (down_stack) using tf.keras.Model
down_stack = tf.keras.Model(inputs=base_model.input, outputs=base_model_outputs)

# Freeze the pre-trained layers to avoid training them during fine-tuning
down_stack.trainable = False

# Verify the model summary to ensure correct structure
down_stack.summary()

In [ ]:
# Use pix2pix upsample layers to get our unsampling block
up_stack = [
    pix2pix.upsample(512, 3),  # 4x4 -> 8x8
    pix2pix.upsample(256, 3),  # 8x8 -> 16x16
    pix2pix.upsample(128, 3),  # 16x16 -> 32x32
    pix2pix.upsample(64, 3),   # 32x32 -> 64x64
]

In [ ]:
def unet_model(output_channels:int):
  inputs = keras.Input(shape=(128, 128, 3))# Create an Input Layer of shape (128,128,3)

  # Downsampling through the model
  skips = down_stack(inputs)
  x = skips[-1]
  skips = reversed(skips[:-1])

  # Upsampling and establishing the skip connections
  for up, skip in zip(up_stack, skips):
    x = up(x)
    x = layers.Concatenate()([x, skip])

  # This is the last layer of the model
  last = tf.keras.layers.Conv2DTranspose(
      filters=output_channels, kernel_size=3, strides=2,
      padding='same')  #64x64 -> 128x128

  x = last(x)

  return tf.keras.Model(inputs=inputs, outputs=x)

In [ ]:
OUTPUT_CLASSES = 3

model = unet_model(output_channels=OUTPUT_CLASSES)
model.summary()

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.01),
    loss=SparseCategoricalCrossentropy(from_logits=True),  # Use SparseCategoricalCrossentropy for scalar integer labels
    metrics=['accuracy']
)

In [ ]:
plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)
img = plt.imread('model_architecture.png')
plt.figure(figsize=(12, 12))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
def create_mask(pred_mask):
  pred_mask = tf.math.argmax(pred_mask, axis=-1)
  pred_mask = pred_mask[..., tf.newaxis]
  return pred_mask[0]

In [ ]:
def show_predictions(dataset=None, num=1):
  if dataset:
    for image, mask in dataset.take(num):
      sample_image = image[0]
      sample_mask = mask[0]
      pred_mask = model.predict(image)
      display([image[0], mask[0], create_mask(pred_mask)])
  else:
    display([sample_image, sample_mask,
             create_mask(model.predict(sample_image[tf.newaxis, ...]))])

In [ ]:
def show_predictions(dataset=None, num=1):
    for image, mask in dataset.take(num):
        # Pick a sample image and mask from the dataset
        sample_image = image[0]
        sample_mask = mask[0]

        # Get the model's prediction for this image
        pred_mask = model.predict(sample_image[tf.newaxis, ...])

        # Display the image, mask, and predicted mask
        display([sample_image, sample_mask, create_mask(pred_mask)])

In [ ]:
show_predictions(train_batches)

In [ ]:
class DisplayCallback(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs=None):
    clear_output(wait=True)
    show_predictions(train_batches)
    print ('\nSample Prediction after epoch {}\n'.format(epoch+1))

In [ ]:
EPOCHS = 20
VAL_SUBSPLITS = 5
VALIDATION_STEPS = info.splits['test'].num_examples//BATCH_SIZE//VAL_SUBSPLITS

model_history = model.fit(train_batches, epochs=EPOCHS,
                          steps_per_epoch=STEPS_PER_EPOCH,
                          validation_steps=VALIDATION_STEPS,
                          validation_data=test_batches,
                          callbacks=[DisplayCallback()])

In [ ]:
# Plotting Loss
epochs = range(1, len(model_history.history["loss"]) + 1)
loss = model_history.history["loss"] # history me se Loss nikalo
val_loss = model_history.history["val_loss"] # history me se validation loss nikalo
plt.figure()
plt.plot(epochs, loss, "bo", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()

In [ ]:
show_predictions(test_batches, 20)

# OBJECT DETECTION

In [ ]:
import os
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import clear_output
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import load_img, img_to_array, array_to_img, plot_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import tensorflow_datasets as tfds
import gc
import cv2 as cv

In [ ]:
def list_files(full_data_path = "/kaggle/input/labeled-mask-dataset-yolo-darknet/obj", image_ext = '.jpg', split_percentage = [70, 20]):

    files = []

    discarded = 0
    masked_instance = 0

    for r, d, f in os.walk(full_data_path):
        for file in f:
            if file.endswith(".txt"):

                # first, let's check if there is only one object
                with open(full_data_path + "/" + file, 'r') as fp:
                    lines = fp.readlines()
                    if len(lines) > 1:
                        discarded += 1
                        continue


                strip = file[0:len(file) - len(".txt")]
                # secondly, check if the paired image actually exist
                image_path = full_data_path + "/" + strip + image_ext
                if os.path.isfile(image_path):
                    # checking the class. '0' means masked, '1' for unmasked
                    if lines[0][0] == '0':
                        masked_instance += 1
                    files.append(strip)

    size = len(files)
    print(str(discarded) + " file(s) discarded")
    print(str(size) + " valid case(s)")
    print(str(masked_instance) + " are masked cases")

    random.shuffle(files)

    split_training = int(split_percentage[0] * size / 100)
    split_validation = split_training + int(split_percentage[1] * size / 100)

    return files[0:split_training], files[split_training:split_validation], files[split_validation:]

training_files, validation_files, test_files = list_files()

In [ ]:
print(str(len(training_files)) + " training files")
print(str(len(validation_files)) + " validation files")
print(str(len(test_files)) + " test files")

In [ ]:
input_size = 244

def format_image(image, box):
    height = image.shape[0]
    width = image.shape[1]
    max_dim = max(height, width)
    scaling_factor = input_size/max_dim
    new_height = int(height * scaling_factor)
    new_width = int(width * scaling_factor)
    new_shape = (new_height, new_width)
    img = cv.resize(image, new_shape, interpolation = cv.INTER_LINEAR)
    new_img = np.zeros((input_size, input_size))
    new_img[:new_width, :new_height] = img[:, :]
    x, y, w, h = box[0], box[1], box[2], box[3]
    new_box = [x*scaling_factor, y*scaling_factor, w*scaling_factor, h*scaling_factor]
    return new_img,new_box

In [ ]:
def data_load(files, full_data_path='/kaggle/input/labeled-mask-dataset-yolo-darknet/obj', img_ext='.jpg'):
    X = []
    Y = []
    for file in files:
        img = cv.imread(os.path.join(full_data_path, file + img_ext), cv.IMREAD_GRAYSCALE)
        k=1
        with open(full_data_path + "/" + file + ".txt") as fp:
            line = fp.readlines()[0]
            if line[0] == "0":
                k=0
            box = np.array(line[1:].split(), dtype=float)
        img, box = format_image(img, box)
        img = img.astype(float) / 255.
        box = np.asarray(box, dtype=float) / input_size
        label = np.append(box, k)
        X.append(img)
        Y.append(label)
    X = np.array(X)
    X = np.expand_dims(X, axis=3)
    X = tf.convert_to_tensor(X, dtype=float)
    Y = tf.convert_to_tensor(Y, dtype=float)
    return tf.data.Dataset.from_tensor_slices((X, Y))

In [ ]:
raw_train_ds = data_load(training_files)

In [ ]:
raw_validation_ds = data_load(validation_files)

In [ ]:
raw_test_ds = data_load(test_files)

In [ ]:
CLASSES = 2
def format_instance(image, label):
    print(label[0], label[1], label[2], label[3])
    return image, (tf.one_hot(int(label[4]), CLASSES), [label[0], label[1], label[2], label[3]])

In [ ]:
BATCH_SIZE = 32

def tune_training_ds(dataset):
    dataset = dataset.map(format_instance, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(1024, reshuffle_each_iteration=True)
    dataset = dataset.repeat() # The dataset be repeated indefinitely.
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
for X_batch, Y_batch in train_ds.take(1):  # Use .take(1) to fetch just one batch
    # Y_batch will contain both the one-hot encoded labels and bounding box information
    print(Y_batch)

    # If you want to access specific parts of Y_batch:
    one_hot_labels = Y_batch[0]  # This is the one-hot encoded label (for mask or no mask)
    bbox = Y_batch[1]            # This is the bounding box information (the four coordinates)

    print("One-hot label:", one_hot_labels)
    print("Bounding box:", bbox)

In [ ]:
def tune_validation_ds(dataset):
    dataset = dataset.map(format_instance, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(len(validation_files) // 4)
    dataset = dataset.repeat()
    return dataset

In [ ]:
validation_ds = tune_validation_ds(raw_validation_ds)

In [ ]:
for X_batch, Y_batch in validation_ds.take(1):  # Use .take(1) to fetch just one batch
    # Y_batch will contain both the one-hot encoded labels and bounding box information
    print(Y_batch)

    # If you want to access specific parts of Y_batch:
    one_hot_labels = Y_batch[0]  # This is the one-hot encoded label (for mask or no mask)
    bbox = Y_batch[1]            # This is the bounding box information (the four coordinates)

    print("One-hot label:", one_hot_labels)
    print("Bounding box:", bbox)

In [ ]:
def tune_test_ds(dataset):
    dataset = dataset.map(format_instance, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(1)
    dataset = dataset.repeat()
    return dataset

In [ ]:
for X_batch, Y_batch in train_ds.take(1):  # Use .take(1) to fetch just one batch
    # Y_batch will contain both the one-hot encoded labels and bounding box information    print(Y_batch)

    # If you want to access specific parts of Y_batch:
    one_hot_labels = Y_batch[0]  # This is the one-hot encoded label (for mask or no mask)
    bbox = Y_batch[1]            # This is the bounding box information (the four coordinates)

    print("One-hot label:", one_hot_labels)
    print("Bounding box:", bbox)

In [ ]:
test_ds = tune_test_ds(raw_test_ds)

In [ ]:
for X_batch, Y_batch in test_ds.take(1):  # Use .take(1) to fetch just one batch
    # Y_batch will contain both the one-hot encoded labels and bounding box information
    print(Y_batch)

    # If you want to access specific parts of Y_batch:
    one_hot_labels = Y_batch[0]  # This is the one-hot encoded label (for mask or no mask)
    bbox = Y_batch[1]            # This is the bounding box information (the four coordinates)

    print("One-hot label:", one_hot_labels)
    print("Bounding box:", bbox)

In [ ]:
def build_feature_extractor(inputs):
    # use 3 pairs of Conv2D and AveragePooling2D with relu activation and kernel size = 3. Keep in mind we will be using
    x = layers.Conv2D(32, kernel_size=3, padding='same', activation='relu')(inputs)
    x = layers.AveragePooling2D(pool_size=(2, 2), strides=2)(x)
    x = layers.Conv2D(64, kernel_size=3, padding='same', activation='relu')(inputs)
    x = layers.AveragePooling2D(pool_size=(2, 2), strides=2)(x)
    x = layers.Conv2D(128, kernel_size=3, padding='same', activation='relu')(inputs)
    x = layers.AveragePooling2D(pool_size=(2, 2), strides=2)(x)
    return x

def build_model_adaptor(inputs):
    # Use one Flatten and One Dense Relu Layer
    x = layers.Flatten()(inputs)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dense(256, activation='relu')(x)
    return x

def build_classifier_head(inputs):
    # use a Softmax Layer named classifier_head
    classifier_head = layers.Dense(2, activation='softmax')(inputs)
    return classifier_head

def build_regressor_head(inputs):
    # use a Dense layer with 4 units named regressor_head
    regressor_head = layers.Dense(4, activation='relu')(inputs)
    return regressor_head

def build_model(inputs):

    feature_extractor = build_feature_extractor(inputs)

    model_adaptor = build_model_adaptor(feature_extractor)

    classification_head = build_classifier_head(model_adaptor)

    regressor_head = build_regressor_head(model_adaptor)

    model = tf.keras.Model(inputs = inputs, outputs = [classification_head, regressor_head])

    return model

In [ ]:
model = build_model(tf.keras.layers.Input(shape=(input_size, input_size, 1,)))

model.compile(optimizer=Adam(learning_rate=0.01), loss=['binary_crossentropy', 'mean_squared_error'], metrics=['accuracy', 'accuracy'])# Use Adam and set loss and metric for classifier_head and regressor_head as stated earlier

In [ ]:
model.summary()

In [ ]:
plot_model(model, to_file='model_architecture.png', show_shapes=True, show_layer_names=True)
img = plt.imread('model_architecture.png')
plt.figure(figsize=(12, 12))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
EPOCHS = 100
BATCH_SIZE = 32

history = model.fit(train_ds,
                    steps_per_epoch=(len(training_files) // BATCH_SIZE),
                    validation_data=validation_ds, validation_steps=1,
                    epochs=EPOCHS)

In [ ]:
def intersection_over_union(boxA, boxB):
    # Calculate the coordinates of the intersection area
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    
    # Calculate intersection area
    interArea = (xB - xA) * (yB - yA)  # Intersection Area
    
    # Calculate individual box areas
    boxAArea = boxA[2] * boxA[3]  # Area of Box A
    boxBArea = boxB[2] * boxB[3]  # Area of Box B
    
    # If either box has zero area, return IoU as 0
    if boxAArea == 0 and boxBArea == 0:
        print("case 1")
        return 0.0
    
    # Calculate the IoU (intersection over union)
    unionArea = boxAArea + boxBArea - interArea
    if unionArea == 0:  # If no union (i.e., boxes don't overlap)
        print("case 2")
        return 0.0
    
    iou = interArea / unionArea
    return iou


In [ ]:
plt.figure(figsize=(12, 10))

test_list = list(test_ds.take(20).as_numpy_iterator())

image, labels = test_list[0]

for i in range(len(test_list)):

    ax = plt.subplot(4, 5, i + 1)
    image, labels = test_list[i]

    predictions = model(image)

    predicted_box = predictions[1][0] * input_size
    predicted_box = tf.cast(predicted_box, tf.int32)
    print(predicted_box)
    predicted_label = predictions[0][0]

    image = image[0]

    actual_label = labels[0][0]
    actual_box = labels[1][0] * input_size
    actual_box = tf.cast(actual_box, tf.int32)
    print(actual_box)

    image = image.astype("float") * 255.0
    image = image.astype(np.uint8)
    image_color = cv.cvtColor(image, cv.COLOR_GRAY2RGB)

    color = (255, 0, 0)
    # print box red if predicted and actual label do not match
    if (predicted_label[0] > 0.5 and actual_label[0] > 0) or (predicted_label[0] < 0.5 and actual_label[0] == 0):
        color = (0, 255, 0)

    img_label = "unmasked"
    if predicted_label[0] > 0.5:
        img_label = "masked"

    predicted_box_n = predicted_box.numpy()
    cv.rectangle(image_color, predicted_box_n, color, 2)
    cv.rectangle(image_color, actual_box.numpy(), (0, 0, 255), 2)
    cv.rectangle(image_color, (predicted_box_n[0], predicted_box_n[1] + predicted_box_n[3] - 20), (predicted_box_n[0] + predicted_box_n[2], predicted_box_n[1] + predicted_box_n[3]), color, -1)
    cv.putText(image_color, img_label, (predicted_box_n[0] + 5, predicted_box_n[1] + predicted_box_n[3] - 5), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0))

    IoU = intersection_over_union(predicted_box.numpy(), actual_box.numpy())

    plt.title("IoU:" + format(IoU, '.4f'))
    plt.imshow(image_color)
    plt.axis("off")

In [ ]:
test_list[0][0][0][0]

# EFFICIENTNETB3 MODEL

In [ ]:
import os
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import clear_output
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import load_img, img_to_array, array_to_img, plot_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
import tensorflow_datasets as tfds
import gc
import cv2 as cv

In [ ]:
data_dir = "/kaggle/input/pins-face-recognition/105_classes_pins_dataset"

In [ ]:
img_height, img_width = 180, 180
batch_size = 32

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    label_mode='categorical',
    image_size=(img_height, img_width),
    batch_size=batch_size
)

In [ ]:
img_height, img_width = 180,180
batch_size = 32

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    label_mode='categorical',
    image_size=(img_height, img_width),
    batch_size=batch_size
)

In [ ]:
base_model = tf.keras.applications.EfficientNetB3(
    include_top=False,
    weights='imagenet',
    input_shape=(180, 180, 3) #image shape here
)

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),
    layers.RandomTranslation(0.1, 0.1),
])

In [ ]:
inputs = layers.Input(shape=(180, 180, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x) ##what is GAP
x = layers.Dropout(0.5)(x)
x = layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001))(x)
x = layers.BatchNormalization()(x)
outputs = layers.Dense(105, activation='softmax')(x)

model = keras.Model(inputs, outputs)

In [ ]:
def lr_schedule(epoch):
    # tweak this around if you want to
    
    lr = 1e-3
    if epoch > 10:
        lr *= 0.1
    if epoch > 20:
        lr *= 0.1
    return lr

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

In [ ]:
initial_epochs = 20
callbacks = [
    keras.callbacks.LearningRateScheduler(lr_schedule),
    keras.callbacks.ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
]

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=initial_epochs,
    callbacks=callbacks
)

In [ ]:
# Fine-tuning
base_model.trainable = True

# Freeze batch norm layers
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
# Recompile the model
model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

In [ ]:
# fine_tune_epochs = 30 total_epochs = initial_epochs + fine_tune_epochs

history_fine = model.fit( train_ds, validation_data=val_ds, epochs=30, initial_epoch=initial_epochs, callbacks=callbacks )

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pickle

# Get class names from the dataset
class_names = train_ds.class_names

# Create and fit LabelEncoder
le = LabelEncoder()
le.fit(class_names)

# Save LabelEncoder
with open('label_encoder.pkl', 'wb') as le_file:
    pickle.dump(le, le_file)

print("LabelEncoder saved successfully.")

model.save('/kaggle/working/efficientnetb3.keras')
print("Final model saved successfully.")

In [ ]:
from IPython.display import FileLink

FileLink('/kaggle/working/efficientnetb3.keras')  # Click on the link to download

In [ ]:
!zip -r efficientnetb3.zip "/kaggle/working/efficientnetb3.keras"